# AE2 - Modelo Oculto de Markov (HMM)
## Riesgo de incendio forestal

Este notebook ejecuta el proyecto completo: define el HMM, simula una temporada de 120 días, decodifica los estados ocultos con **Viterbi**, calcula probabilidades posteriores con **forward-backward**, genera alertas de quemas y produce gráficos y tablas.

**Estados ocultos:** Riesgo Bajo · Riesgo Moderado · Riesgo Extremo  
**Observaciones:** Humedad Alta · Humedad Media · Humedad Baja

## 0. Preparación del entorno

El notebook busca la raíz del proyecto (la carpeta que contiene `src/`) y la agrega al `sys.path`, de modo que funciona aunque se abra desde otra ubicación.

In [ ]:
%matplotlib inline

from pathlib import Path
import sys

ROOT = Path.cwd().resolve()
for candidate in (ROOT, *ROOT.parents):
    if (candidate / "src" / "hmm_model.py").exists():
        ROOT = candidate
        break
else:
    raise FileNotFoundError(
        "No se encontró la carpeta src/. Abra el notebook desde la raíz del proyecto."
    )

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from src.hmm_model import HMM
from src.simulation import compute_metrics, run_simulation
from src.utils import (
    format_report,
    plot_danger_index_and_comparison,
    plot_humidity_and_risk,
    plot_summary_dashboard,
    results_dataframe,
)

print(f"Raíz del proyecto: {ROOT}")
print(f"NumPy {np.__version__} | pandas {pd.__version__} | matplotlib {plt.matplotlib.__version__}")

## 1. Parámetros de la simulación

Puede modificar estos valores y volver a ejecutar las celdas siguientes.

In [ ]:
N_DAYS = 120
SEED = 42
ALERT_THRESHOLD = 75.0
OUTPUT_DIR = ROOT / "outputs"

## 2. Matrices del modelo

- **A**: matriz de transición entre estados ocultos.
- **B**: matriz de emisión de observaciones (humedad).
- **π**: distribución inicial de estados.

Cada fila de A y B debe sumar 1; el modelo valida esto automáticamente.

In [ ]:
model = HMM(seed=SEED)

print("Matriz de transición A")
display(pd.DataFrame(model.A, index=model.states, columns=model.states).round(2))

print("Matriz de emisión B")
display(pd.DataFrame(model.B, index=model.states, columns=model.observations).round(2))

print("Distribución inicial pi")
display(pd.DataFrame({"Estado": model.states, "Probabilidad": model.pi}))

assert np.allclose(model.A.sum(axis=1), 1.0)
assert np.allclose(model.B.sum(axis=1), 1.0)
assert np.isclose(model.pi.sum(), 1.0)
print("Validación estocástica: OK")

## 3. Simulación de la temporada y decodificación

Se generan los estados reales y las observaciones de humedad, se aplica Viterbi y forward-backward, y se calculan las métricas operativas.

In [ ]:
simulation = run_simulation(
    model=model,
    n_days=N_DAYS,
    seed=SEED,
    alert_threshold=ALERT_THRESHOLD,
)
metrics = compute_metrics(simulation)

print(format_report(simulation, metrics))

## 4. Tabla diaria de resultados

In [ ]:
daily = results_dataframe(simulation)
print("Primeros 8 días:")
display(daily.head(8))

print("Últimos 8 días:")
daily.tail(8)

## 5. Alertas de quema

`PROHIBICIÓN TOTAL` se activa cuando el estado inferido es de riesgo extremo; `RESTRICCIÓN PREVENTIVA` cuando el índice de peligro supera el umbral configurado.

In [ ]:
alerts = pd.DataFrame(simulation.alerts)
if alerts.empty:
    print("No se activaron alertas en la temporada simulada.")
else:
    display(alerts[['day', 'level', 'danger_index']].rename(columns={
        'day': 'Día',
        'level': 'Nivel de alerta',
        'danger_index': 'Índice de peligro',
    }))

## 6. Gráficos

Las figuras se muestran en el notebook y se guardan en `outputs/`.

In [ ]:
OUTPUT_DIR.mkdir(exist_ok=True)

fig_humidity = plot_humidity_and_risk(
    simulation, save_path=OUTPUT_DIR / "humedad_y_riesgo.png"
)
plt.show()

In [ ]:
fig_danger = plot_danger_index_and_comparison(
    simulation, save_path=OUTPUT_DIR / "indice_y_comparacion.png"
)
plt.show()

In [ ]:
fig_summary = plot_summary_dashboard(
    simulation, save_path=OUTPUT_DIR / "resumen_metricas.png"
)
plt.show()

## 7. Exportación de resultados

Se guarda la tabla diaria en CSV junto con las tres figuras.

In [ ]:
csv_path = OUTPUT_DIR / "resultados_diarios.csv"
daily.to_csv(csv_path, index=False)

print(f"Resultados guardados en: {OUTPUT_DIR.resolve()}")
for path in sorted(OUTPUT_DIR.iterdir()):
    if path.is_file():
        print(f"  - {path.name} ({path.stat().st_size / 1024:.1f} KB)")

## 8. Resumen final

Métricas clave de la temporada simulada.

In [ ]:
summary = pd.DataFrame(
    {
        "Métrica": [
            "Días simulados",
            "Exactitud de Viterbi",
            "Días en riesgo extremo (inferido)",
            "Índice de peligro medio",
            "Índice de peligro máximo",
            "Días con alerta",
            "Focos simulados",
        ],
        "Valor": [
            simulation.n_days,
            f"{metrics['comparison']['accuracy']:.2%}",
            metrics["extreme_days_inferred"],
            f"{metrics['mean_danger_index']:.1f}/100",
            f"{metrics['max_danger_index']:.1f}/100",
            metrics["alert_count"],
            metrics["humidity_fire"]["total_fire_events"],
        ],
    }
)
summary